# Codex-powered OPAL pilot

The GRU architecture remains unchanged. OPAL changes optimizer programs during training.
The proposed language model is gpt-6.1-sol with high effort.
Use the optional setup on a fresh machine. Preflight never starts an agent.
All training cells are skipped during report export, even when flags are enabled.
Read docs/opal-pilot.md before enabling any training flag.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("gru_suite_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/gru-opal-codex.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/opal-codex-pilot"  # Select the PR branch before merge
RUN_SETUP = False
RUN_KEEP = False
RUN_CONTROL = False
RUN_LIVE = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Keep training enabled: %s", RUN_KEEP)
logger.info("Control training enabled: %s", RUN_CONTROL)
logger.info("Live training enabled: %s", RUN_LIVE)

## Optional setup

In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group opal --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without optimization

Validate data and tensor shapes. This does not check Codex authentication or launch OPAL.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook --group opal python -m sleep_classifier.gru_training \
    --config configs/gru-opal-codex.yaml --check
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Pilot preflight failed")

## Fresh Adam control

Enable RUN_CONTROL only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_CONTROL:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-control.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Fresh Adam control failed")

## Offline keep comparison

Enable RUN_KEEP only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_KEEP:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-keep.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Offline keep comparison failed")

## Live Codex pilot

Enable RUN_LIVE only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_LIVE:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-codex.yaml --train --allow-live-agent
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Live Codex pilot failed")

## Controlled retry with a pinned CLI

The original Codex attempt failed its context audit and applied no optimizer changes.
Do not treat its accuracy as a successful reactive-optimization result.
Install the pinned stable CLI before the retry. This does not alter the app or Python dependencies.
The retry uses new output folders, one decision at update 50, and the same model settings.
Live compatibility remains unverified until a decision passes the unchanged isolation audit.
The token allowance remains soft. Read docs/opal-compatibility-notes.md before enabling a retry.


In [ ]:
RUN_INSTALL_STABLE_CLI = False
RUN_RETRY_CONTROL = False
RUN_RETRY_KEEP = False
RUN_RETRY_LIVE = False

In [ ]:
if RUN_INSTALL_STABLE_CLI:
    %cd "{ROOT}"
    !npm ci --prefix tools/opal-codex --no-audit --no-fund
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Pinned CLI installation failed")

In [ ]:
!uv run --locked --group gru --group notebook --group opal python \
    -m sleep_classifier.gru_training --config configs/gru-opal-retry-codex.yaml --check-agent
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Retry runtime is not ready")

In [ ]:
if RUN_RETRY_CONTROL:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-retry-control.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("control retry failed; inspect the run logs")

In [ ]:
if RUN_RETRY_KEEP:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-retry-keep.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("keep retry failed; inspect the run logs")

In [ ]:
if RUN_RETRY_LIVE:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-retry-codex.yaml --train --allow-live-agent
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("codex retry failed; inspect the run logs")

## Inspect saved metrics

Inspect OPAL programs, applied decisions, wait times, and token counts before interpreting any live score.

In [ ]:
import pandas as pd
import yaml

from sleep_classifier.opal_report import OpalRunReport

records = []
run_reports = []
recipes = [f"gru-opal-{kind}" for kind in ["control", "keep", "codex"]]
recipes += [f"gru-opal-retry-{kind}" for kind in ["control", "keep", "codex"]]
for recipe in recipes:
    config_path = ROOT / f"configs/{recipe}.yaml"
    config = yaml.safe_load(config_path.read_text())
    run_dir = (config_path.parent / config["output_dir"]).resolve()
    report = OpalRunReport(run_dir).build()
    if report["metrics_available"] or report["events_available"]:
        run_reports.append((recipe, report))
        records.append({"recipe": recipe, "status": report["status"],
                        **report["classification"],
                        "applied": (report["decision_counts"] or {}).get("applied"),
                        "tokens": report["usage"].get("total_tokens"),
                        "wait_seconds": report["agent_wait_seconds"],
                        "soft_cap_exceeded": report["soft_token_cap_exceeded"],
                        "checkpoint_before_change": report.get("checkpoint_precedes_first_change")})
        logger.info("Optimizer status for %s: %s", recipe, report["status"])
        display(pd.DataFrame([{
            "end_update": report["end_update"],
            "selected_checkpoint_update": report.get("selected_checkpoint_update"),
            "first_applied_update": report["first_applied_update"],
            "final_program_version": report["final_program_version"],
            "unreached_decisions": report["unreached_scheduled_updates"],
        }]))
        display(pd.DataFrame([report["decision_counts"] or {}]))
        display(pd.DataFrame([report["usage"]]))
        display(pd.DataFrame([report["configured_agent"]]))
        if report["runtime"]:
            display(pd.DataFrame([report["runtime"]]))
        if report["decisions"]:
            display(pd.DataFrame(report["decisions"]))
            for decision in report["decisions"]:
                if decision.get("backend_error"):
                    logger.error("Decision %s failed: %s", decision["k"],
                                 decision["backend_error"])
if records:
    display(pd.DataFrame(records))
else:
    logger.info("No pilot results exist. Preflight did not train.")

In [ ]:
if run_reports:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for recipe, report in run_reports:
        config_path = ROOT / f"configs/{recipe}.yaml"
        config = yaml.safe_load(config_path.read_text())
        metrics_path = (config_path.parent / config["output_dir"] / "metrics.json").resolve()
        if metrics_path.exists():
            history = pd.DataFrame(json.loads(metrics_path.read_text())["history"])
            axes[0].plot(history.epoch, history.validation_macro_f1, label=recipe)
    axes[0].set(xlabel="Epoch", ylabel="Validation macro-F1", ylim=(0, 1))
    axes[0].legend(fontsize=7)
    labels = [recipe for recipe, report in run_reports
              if report["usage"].get("total_tokens") is not None]
    tokens = [report["usage"]["total_tokens"] for recipe, report in run_reports
              if report["usage"].get("total_tokens") is not None]
    axes[1].bar(labels, tokens)
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].set(ylabel="Reported tokens, including cached input")
    figure.tight_layout()
    plt.show()

## Export HTML

In [ ]:
!uv run --locked --group gru --group notebook --group opal python scripts/execute_notebook.py \
    --notebook notebooks/10_opal_pilot.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Report export failed")